# 03 — Unification des sources · Databricks
**Exécuter après** : `02_cleaning_db.ipynb`
---

In [0]:
# ============================================================
# Setup Databricks — import spark_utils par chemin absolu
# Compatible /Users/ et /Repos/ (Databricks Workspace moderne)
# ============================================================
import sys, os, importlib.util

# 1. Résolution du repo root
#    Le notebook est dans .../mon-repo/databricks/nom_notebook
#    On remonte d'un niveau par rapport au dossier 'databricks'
_ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_parts = _ctx.split('/')

# Chercher le dossier 'databricks' dans le path et prendre tout ce qui est avant
_db_idx = next((i for i, p in enumerate(_parts) if p == 'databricks'), None)
if _db_idx is not None:
    _REPO = '/Workspace/' + '/'.join(_parts[1:_db_idx])
else:
    # Fallback : remonter 2 niveaux depuis le notebook
    _REPO = '/Workspace/' + '/'.join(_parts[1:-2])

_UTILS_FILE = _REPO + '/02_preprocessing/spark_utils.py'

print(f'Notebook path : {_ctx}')
print(f'Repo root     : {_REPO}')
print(f'spark_utils   : {_UTILS_FILE}')

if not os.path.exists(_UTILS_FILE):
    raise FileNotFoundError(
        f"spark_utils.py introuvable : {_UTILS_FILE}"

        f"Verifier que le repo contient bien 02_preprocessing/spark_utils.py"
    )

# 2. Import direct par chemin absolu (bypass du package pip 'spark_utils')
_spec = importlib.util.spec_from_file_location('_spark_utils', _UTILS_FILE)
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

load_raw_sources        = _mod.load_raw_sources
show_label_distribution = _mod.show_label_distribution
save_parquet            = _mod.save_parquet
load_parquet            = _mod.load_parquet
stratified_split        = _mod.stratified_split
check_class_balance     = _mod.check_class_balance

# 3. Chemins Volume Unity Catalog
# Trouver le chemin : Databricks > Data > Volumes > clic droit "fakenews" > Copy path
VOLUME        = '/Volumes/workspace/default/fakenews'  # <- MODIFIER si catalog/schema different

RAW_DIR       = VOLUME + '/raw'
PROCESSED_DIR = VOLUME + '/processed'
MODELS_DIR    = VOLUME + '/models'
COLAB_DIR     = VOLUME + '/colab_exports'
REPORTS_DIR   = VOLUME + '/reports'

for d in [PROCESSED_DIR, MODELS_DIR, COLAB_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Volume        : {VOLUME}')
print('Setup OK')

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
print('Imports OK')

## Section 1 — Chargement des fichiers nettoyés

In [0]:
# Lister les parquets cleaned_* sur DBFS
try:
    parquet_items = dbutils.fs.ls(f'{VOLUME}/processed/')
    parquet_files = [
        f.path
        for f in parquet_items
        if f.name.startswith('cleaned_')
    ]
except Exception as e:
    parquet_files = []
    print(f'Erreur DBFS : {e}')

if not parquet_files:
    raise FileNotFoundError(
        f'Aucun fichier cleaned_*.parquet dans {PROCESSED_DIR}.\n'
        'Executer 02_cleaning_db.ipynb en premier.'
    )

print(f'Parquet trouves : {len(parquet_files)}')
dataframes = {}
for p in parquet_files:
    source_name = os.path.basename(p.rstrip('/')).replace('cleaned_', '').replace('.parquet', '')
    df = spark.read.parquet(p)
    dataframes[source_name] = df
    print(f'  Charge : {source_name} — {df.count():,} lignes | colonnes : {df.columns}')

## Section 2 — Harmonisation du schéma commun

In [0]:
unified_dfs = []

for name, df in dataframes.items():
    print(f'\n── Harmonisation : {name}')
    if 'clean_text' not in df.columns:
        text_col = next((c for c in df.columns if 'text' in c.lower()), None)
        if text_col:
            df = df.withColumnRenamed(text_col, 'clean_text')
        else:
            print(f'  Impossible de trouver colonne texte — ignore')
            continue
    if 'label' in df.columns:
        df = df.withColumn('label', F.col('label').cast(IntegerType()))
        df = df.filter(F.col('label').isin(0, 1))
    else:
        print(f'  Pas de colonne label — etiquette REAL (0)')
        df = df.withColumn('label', F.lit(0))
    if 'source' not in df.columns:
        df = df.withColumn('source', F.lit(name))
    df = df.withColumn('text_len', F.size(F.split(F.col('clean_text'), ' ')))
    df = df.select('clean_text', 'label', 'source', 'text_len')
    count = df.count()
    print(f'  {count:,} lignes harmonisees')
    unified_dfs.append(df)

## Section 3 — Union et déduplication

In [0]:
if not unified_dfs:
    raise ValueError('Aucun DataFrame harmonise. Verifiez les etapes precedentes.')

unified = unified_dfs[0]
for df in unified_dfs[1:]:
    unified = unified.union(df)

total = unified.count()
print(f'DataFrame unifie : {total:,} lignes')

before = total
unified = unified.dropDuplicates(['clean_text'])
after = unified.count()
print(f'Deduplication : {before - after:,} doublons supprimes -> {after:,} lignes')

## Section 4 — Rapport de fusion

In [0]:
source_stats = (
    unified.groupBy('source', 'label').count().toPandas()
    .pivot(index='source', columns='label', values='count')
    .fillna(0).astype(int)
)
if 0 in source_stats.columns and 1 in source_stats.columns:
    source_stats.columns = ['REAL', 'FAKE']
    source_stats['total'] = source_stats['REAL'] + source_stats['FAKE']
    source_stats['FAKE%'] = (source_stats['FAKE'] / source_stats['total'] * 100).round(1)
print('=== RAPPORT DE FUSION ===')
display(source_stats)
print(f'TOTAL : {unified.count():,} articles')

## Section 5 — Vérification équilibre + Split train/test

In [0]:
check_class_balance(unified, label_col='label', min_minority_ratio=0.35)
show_label_distribution(unified, label_col='label')

In [0]:
train_df, test_df = stratified_split(unified, label_col='label', test_size=0.2, seed=42)
print(f'Split train/test :')
print(f'  Train : {train_df.count():,}')
print(f'  Test  : {test_df.count():,}')
print('\nDistribution TRAIN :')
show_label_distribution(train_df)
print('\nDistribution TEST :')
show_label_distribution(test_df)

## Section 6 — Sauvegarde

In [0]:
save_parquet(unified,  os.path.join(PROCESSED_DIR, 'unified.parquet'))
save_parquet(train_df, os.path.join(PROCESSED_DIR, 'train.parquet'))
save_parquet(test_df,  os.path.join(PROCESSED_DIR, 'test.parquet'))

print('\nFichiers sauvegardes :')
print(f'  {PROCESSED_DIR}/unified.parquet')
print(f'  {PROCESSED_DIR}/train.parquet')
print(f'  {PROCESSED_DIR}/test.parquet')
print('Prochaine etape : 04_feature_engineering_db.ipynb')